Pretrained CodeT5 based Experiment
Raw Dataset



## Total Object Models: 9


## Training

1.	Camping
2.	Customer_Order
3.	Ecommerce
4.	Onlinestore
5.	Decider
6.  Library OM
7.  CSOS
8.  Flagship

## Testing

9.	Bank





--------------------------------

## Total Training Data: 13236 (100%)
--------------------------------

### Training set P :  9265 (70% of Training Data)

### Training set NP : 3971 (30% of Training Data)

-----------------------------
## Total Testing Data: 32 (100% of Total Data)
----------------------------

### Testing set P : 8 (25% of Testing Data)

### Testing set NP : 24 (75%% of Testing Data)

In [1]:
pip install transformers[torch]

In [2]:
pip install accelerate -U

In [3]:
#Import Libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments
from transformers import RobertaTokenizer, T5ForConditionalGeneration
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import TrainingArguments
import torch
from accelerate import Accelerator

In [4]:
# Initialize accelerator
accelerator = Accelerator()

In [5]:
import pandas as pd

# Load the dataset from the first sheet of the Excel file
data = pd.read_csv("raw_8_om_training_set.csv")

In [6]:
# Selecting the relevant columns
data = data[['OM_Regular', 'OM_Prediction']]

In [7]:
# Splitting data into train and test sets
train_data, test_data = train_test_split(data, test_size=0.3, random_state=42)

In [8]:
# Load the T5 tokenizer and model
# tokenizer = T5Tokenizer.from_pretrained("t5-large")
# tokenizer.model_max_length = 512  # Set maximum sequence length directly in the tokenizer
# model = T5ForConditionalGeneration.from_pretrained("t5-large")

tokenizer = RobertaTokenizer.from_pretrained("Salesforce/codet5-large")
tokenizer.model_max_length = 512
model = T5ForConditionalGeneration.from_pretrained("Salesforce/codet5-large")



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [9]:

   def preprocess_data(data, tokenizer, max_seq_length=512):
    input_texts = list(data['OM_Regular'])
    target_texts = list(data['OM_Prediction'])

    # Tokenize input and target texts
    input_encodings = tokenizer(input_texts, truncation=True, padding='max_length', max_length=max_seq_length)
    target_encodings = tokenizer(target_texts, truncation=True, padding='max_length', max_length=max_seq_length)
        # Convert tokenized sequences to PyTorch tensors
    input_ids = torch.tensor(input_encodings['input_ids'])
    target_ids = torch.tensor(target_encodings['input_ids'])

    return input_ids, target_ids

train_inputs, train_targets = preprocess_data(train_data, tokenizer)
test_inputs, test_targets = preprocess_data(test_data, tokenizer)

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [10]:
# Define a custom dataset class
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return {
            'input_ids': self.inputs[idx],
            'labels': self.targets[idx]
        }


In [11]:
# Create instances of custom dataset
train_dataset = CustomDataset(train_inputs, train_targets)
eval_dataset = CustomDataset(test_inputs, test_targets)


In [12]:
!pip cache purge
!pip install accelerate -U

Files removed: 2


In [13]:

training_args = TrainingArguments(
    output_dir='./results',
     report_to="none",  # Add this line
    num_train_epochs=12,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.02,
    logging_dir='./logs',
    fp16=True,
    logging_steps=500,
    save_steps=1000,
    evaluation_strategy="epoch",
     # Set max_new_tokens here
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [14]:
# Define a Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

)
# Train the model
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.003400,0.001683
2,0.002300,0.001706
3,0.001400,0.001399
4,0.001400,0.001444
5,0.001300,0.001407
6,0.001300,0.001350
7,0.001300,0.001200
8,0.001200,0.001219
9,0.001200,0.001290
10,0.001200,0.001200


TrainOutput(global_step=14004, training_loss=0.018986908575760398, metrics={'train_runtime': 4949.7244, 'train_samples_per_second': 22.629, 'train_steps_per_second': 2.829, 'total_flos': 6.820815540584448e+16, 'train_loss': 0.018986908575760398, 'epoch': 12.0})

In [15]:
# Evaluate the model
eval_results = trainer.evaluate(eval_dataset)

In [16]:
# # Calculate additional metrics
# labels = test_targets.flatten().tolist()
# preds = model.generate(test_inputs)
# preds = torch.tensor(preds).flatten().tolist()
# accuracy = accuracy_score(labels, preds)
# precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

# # Display evaluation results
# print("Evaluation Results:")
# print(f"Accuracy: {accuracy}")
# print(f"Precision: {precision}")
# print(f"Recall: {recall}")
# print(f"F1 Score: {f1}")

In [17]:
# Test the model with new input data
def generate_predictions(input_text, model, tokenizer):
    # Move input text to the same device as the model
    device = next(model.parameters()).device

    # Encode input text
    input_ids = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Generate predictions
    with torch.no_grad():
        outputs = model.generate(input_ids.to('cuda:0'))  # Move input_ids to CPU before generation

    # Move predictions to CPU and decode
    predicted_text = tokenizer.decode(outputs[0].cpu(), skip_special_tokens=True)

    return predicted_text

# Test the model with new input data
input_text = "module ecommerceopen Declarationone sig Customer extends Class{}{attrSet = customerIDid=customerIDisAbstract = Nono parent}one sig customerID extends Integer{}one sig Order extends Class{}{attrSet = orderIDid=orderIDisAbstract = Nono parent}one sig orderID extends Integer{}one sig CustomerOrderAssociation extends Association{}{src = Customerdst = Ordersrc_multiplicity = ONEdst_multiplicity = MANY}one sig ShippingCart extends Class{}{attrSet = shippingCartIDid=shippingCartIDisAbstract = Nono parent}one sig shippingCartID extends Integer{}one sig CustomerShippingCartAssociation extends Association{}{src = Customerdst = ShippingCartsrc_multiplicity = ONEdst_multiplicity = MANY}one sig Item extends Class{}{attrSet = ItemID+quantityid=ItemIDisAbstract = Nono parent}one sig ItemID extends Integer{}one sig quantity extends Integer{}one sig CartItem extends Class{}{attrSet = cartItemIDone parentid=ItemIDisAbstract = Noparent in Item}one sig cartItemID extends Integer{}one sig ShippingCartItemAssociation extends Association{}{src = ShippingCartdst = Itemsrc_multiplicity = ONEdst_multiplicity = MANY}one sig OrderItem extends Class{}{attrSet = orderItemID+statusone parentid=ItemIDisAbstract = Noparent in Item}one sig status extends Integer{}one sig orderItemID extends Integer{}one sig OrderItemAssociation extends Association{}{src = Orderdst = Itemsrc_multiplicity = ONEdst_multiplicity = MANY}one sig Category extends Class{}{attrSet = categoryID+categoryNameid=categoryIDisAbstract = Nono parent}one sig categoryID extends Integer{}one sig categoryName extends string{}one sig Product extends Class{}{attrSet = productID+productName+description+priceid=productIDisAbstract = Nono parent}one sig productID extends Integer{}one sig productName extends string{}one sig description extends string{}one sig price extends Real{}one sig ProductCategoryAssociation extends Association{}{src = Productdst = Categorysrc_multiplicity = MANYdst_multiplicity = MANY}one sig Catalog extends Class{}{attrSet = CatalogIDid=CatalogIDisAbstract = Nono parent}one sig CatalogID extends Integer{}one sig ProductCatalogAssociation extends Association{}{src = Productdst = Catalogsrc_multiplicity = ONEdst_multiplicity = MANY}one sig ProductItemAssociation extends Association{}{src = Productdst = Itemsrc_multiplicity = MANYdst_multiplicity = MANY}one sig PhysicalProduct extends Class{}{attrSet = weight+availabilityone parentid=productIDisAbstract = Noparent in Product}one sig weight extends Real{}one sig availability extends Bool{}one sig ElectronicProduct extends Class{}{attrSet = sizeone parentid=productIDisAbstract = Noparent in Product}one sig size extends string{}one sig Service extends Class{}{attrSet = scheduleone parentid=productIDisAbstract = Noparent in Product}one sig schedule extends string{}one sig Asset extends Class{}{attrSet = assetID+assetName+fileURIid = assetIDisAbstract = Nono parent}one sig assetID extends Integer{}one sig assetName extends string{}one sig fileURI extends string{}one sig ProductAssetAssociation extends Association{}{src = Productdst = Assetsrc_multiplicity = MANYdst_multiplicity = MANY}one sig Media extends Class{}{attrSet = mediaTypeone parentid = assetIDisAbstract = Noparent in Asset}one sig mediaType extends Integer{}one sig Documents extends Class{}{attrSet = excerptone parentid = assetIDisAbstract = Noparent in Asset}one sig excerpt extends string{}pred show{}run show for 48,Table Name: CustomerTable Name: OrderMapping Strategy for Customer : Union Sub ClassMapping Strategy for CartItem : Union Sub ClassMapping Strategy for OrderItem : Union Sub ClassMapping Strategy for Category : Union Sub ClassMapping Strategy for Product : Union Sub ClassMapping Strategy for PhysicalProduct : Union Sub ClassMapping Strategy for Asset : Union Sub ClassMapping Strategy for ElectronicProduct : Joined Sub ClassMapping Strategy for Service : Joined Sub ClassMapping Strategy for Service : Joined Sub ClassMapping Strategy for Documents : Joined Sub ClassAssociation Strategy for ShippingCartItemAssociation : ForeignKeyEmbeddingStrategyAssociation Strategy for CustomerOrderAssociation : OwnAssociationTableStrategyAssociation Strategy for CustomerShippingCartAssociation : OwnAssociationTableStrategyAssociation Strategy for OrderItemAssociation : OwnAssociationTableStrategyAssociation Strategy for ProductCategoryAssociation : OwnAssociationTableStrategyAssociation Strategy for ProductCatalogAssociation : OwnAssociationTableStrategyAssociation Strategy for ProductItemAssociation : OwnAssociationTableStrategyAssociation Strategy for ProductAssetAssociation : OwnAssociationTableStrategy,CREATE TABLE `Order` (`orderID` int NOT NULL,`customerID` int,KEY `FK_Order_customerID_idx` (`customerID`),PRIMARY KEY (`orderID`));CREATE TABLE `Category` (`categoryName` varchar(64),`categoryID` int NOT NULL,PRIMARY KEY (`categoryID`));CREATE TABLE `Customer` (`customerID` int NOT NULL,PRIMARY KEY (`customerID`));CREATE TABLE `Product` (`description` varchar(64),`productName` varchar(64),`price` decimal(20,5),`productID` int NOT NULL,PRIMARY KEY (`productID`));CREATE TABLE `Service` (`schedule` varchar(64),`productID` int NOT NULL,KEY `FK_Service_productID_idx` (`productID`),PRIMARY KEY (`productID`));CREATE TABLE `CartItem` (`cartItemID` int,`quantity` int,`ItemID` int NOT NULL,PRIMARY KEY (`ItemID`));CREATE TABLE `ShippingCart` (`shippingCartID` int NOT NULL,PRIMARY KEY (`shippingCartID`));CREATE TABLE `Catalog` (`CatalogID` int NOT NULL,PRIMARY KEY (`CatalogID`));CREATE TABLE `ProductItemAssociation` (`productID` int NOT NULL,`ItemID` int NOT NULL,KEY `FK_ProductItemAssociation_productID_idx` (`productID`),KEY `FK_ProductItemAssociation_ItemID_idx` (`ItemID`),PRIMARY KEY (`productID`,`ItemID`));CREATE TABLE `Item` (`quantity` int,`ItemID` int NOT NULL,`shippingCartID` int,KEY `FK_Item_shippingCartID_idx` (`shippingCartID`),PRIMARY KEY (`ItemID`));CREATE TABLE `ElectronicProduct` (`size` varchar(64),`description` varchar(64),`productName` varchar(64),`price` decimal(20,5),`productID` int NOT NULL,PRIMARY KEY (`productID`));CREATE TABLE `PhysicalProduct` (`availability` boolean,`weight` decimal(20,5),`productID` int NOT NULL,KEY `FK_PhysicalProduct_productID_idx` (`productID`),PRIMARY KEY (`productID`));CREATE TABLE `CustomerShippingCartAssociation` (`shippingCartID` int NOT NULL,`customerID` int NOT NULL,KEY `FK_CustomerShippingCartAssociation_shippingCartID_idx` (`shippingCartID`),KEY `FK_CustomerShippingCartAssociation_customerID_idx` (`customerID`),PRIMARY KEY (`shippingCartID`,`customerID`));CREATE TABLE `ProductAssetAssociation` (`assetID` int NOT NULL,`productID` int NOT NULL,KEY `FK_ProductAssetAssociation_assetID_idx` (`assetID`),KEY `FK_ProductAssetAssociation_productID_idx` (`productID`),PRIMARY KEY (`assetID`,`productID`));CREATE TABLE `OrderItem` (`orderItemID` int,`status` int,`ItemID` int NOT NULL,KEY `FK_OrderItem_ItemID_idx` (`ItemID`),PRIMARY KEY (`ItemID`));CREATE TABLE `Asset` (`DType` varchar(64),`excerpt` varchar(64),`fileURI` varchar(64),`assetName` varchar(64),`mediaType` int,`assetID` int NOT NULL,PRIMARY KEY (`assetID`));CREATE TABLE `ProductCatalogAssociation` (`CatalogID` int NOT NULL,`productID` int NOT NULL,KEY `FK_ProductCatalogAssociation_CatalogID_idx` (`CatalogID`),KEY `FK_ProductCatalogAssociation_productID_idx` (`productID`),PRIMARY KEY (`CatalogID`,`productID`));CREATE TABLE `OrderItemAssociation` (`ItemID` int NOT NULL,`orderID` int NOT NULL,KEY `FK_OrderItemAssociation_ItemID_idx` (`ItemID`),KEY `FK_OrderItemAssociation_orderID_idx` (`orderID`),PRIMARY KEY (`ItemID`,`orderID`));CREATE TABLE `ProductCategoryAssociation` (`productID` int NOT NULL,`categoryID` int NOT NULL,KEY `FK_ProductCategoryAssociation_productID_idx` (`productID`),KEY `FK_ProductCategoryAssociation_categoryID_idx` (`categoryID`),PRIMARY KEY (`productID`,`categoryID`));"
predicted_text = generate_predictions(input_text, model, tokenizer)
print("Predicted Output:", predicted_text)




/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1375: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Predicted Output: The object-relational strategy represented by this sequence is the Pareto-optimal


In [18]:
# Test the model with new input data
input_text_2 = "module CSOSopen Declarationone sig Channel extends Class{}{attrSet = channelIDid=channelIDisAbstract = Nono parent}one sig channelID extends Integer{}one sig EmailChannel extends Class{}{attrSet = emailIDone parentparent in ChannelisAbstract = Noid=channelID}one sig emailID extends Integer{}one sig SecEmailChannel extends Class{}{attrSet = secEmailIDone parentparent in EmailChannelisAbstract = Noid=channelID}one sig secEmailID extends Integer{}one sig SMSChannel extends Class{}{attrSet = smsProvider + telNoone parentparent in ChannelisAbstract = Noid=channelID}one sig smsProvider extends string{}one sig telNo extends string{}one sig Principal extends Class{}{attrSet = principalID + principalNameid=principalIDisAbstract = Nono parent}one sig principalID extends Integer{}one sig principalName extends string{}one sig Person extends Class{}{attrSet = roleone parentparent in PrincipalisAbstract = Noid=principalID}one sig role extends Integer{}one sig Viewer extends Class{}{attrSet = periodid=principalIDone parentparent in PersonisAbstract = No}one sig period extends Integer{}one sig Institution extends Class{}{attrSet = InstitutionIDone parentparent in PrincipalisAbstract = Noid=principalID}one sig InstitutionID extends Integer{}one sig PrincipalProxy extends Association{}{src = Principaldst = Channelsrc_multiplicity = ONEdst_multiplicity = MANY}one sig ProcessStateMachine extends Class{}{attrSet = stateMachineIDid=stateMachineIDisAbstract = Nono parent}one sig stateMachineID extends Integer{}one sig ProcessStateMachineState extends Class{}{attrSet = stateID + entryActionID + exitActionIDid=stateIDisAbstract = Nono parent}one sig stateID extends Integer{}one sig entryActionID extends Integer{}one sig exitActionID extends Integer{}one sig ProcessStateMachineAction extends Class{}{attrSet = actionID + actionStateMachineIDid=actionIDisAbstract = Nono parent}one sig actionID extends Integer{}one sig actionStateMachineID extends Integer{}one sig StateMachineStates extends Association{}{src = ProcessStateMachinedst = ProcessStateMachineStatesrc_multiplicity = ONEdst_multiplicity = MANY}one sig ProcessStateMachineEvent extends Class{}{attrSet = eventIDid=eventIDisAbstract = Nono parent}one sig eventID extends Integer{}one sig StateMachineEvents extends Association{}{src = ProcessStateMachinedst = ProcessStateMachineEventsrc_multiplicity = ONEdst_multiplicity = MANY}one sig ProcessStateMachineTransition extends Class{}{attrSet = transitionID + fromStateID + toStateIDid=transitionIDisAbstract = Nono parent}one sig transitionID extends Integer{}one sig fromStateID extends Integer{}one sig toStateID extends Integer{}one sig StateMachineTransitions extends Association{}{src = ProcessStateMachinedst = ProcessStateMachineTransitionsrc_multiplicity = ONEdst_multiplicity = MANY}one sig ProcessStateMachineExecution extends Class{}{attrSet = processStateMachineExecutionID + processStateMachineID + currentStateIDid=processStateMachineExecutionIDisAbstract = Nono parent}one sig processStateMachineExecutionID extends Integer{}one sig processStateMachineID extends Integer{}one sig currentStateID extends Integer{}pred show{}run show for 43,Mapping Strategy for class1_name : map_str2Mapping Strategy for class6_name : map_str2Mapping Strategy for class7_name : map_str2Mapping Strategy for class8_name : map_str2Mapping Strategy for class5_name : map_str3Association Strategy for assoc3 : assoc_str1Association Strategy for assoc8 : assoc_str1Association Strategy for assoc1 : assoc_str2Association Strategy for assoc10 : assoc_str2Association Strategy for assoc4 : assoc_str2Association Strategy for assoc5 : assoc_str2Association Strategy for assoc6 : assoc_str2Association Strategy for assoc7 : assoc_str2Association Strategy for assoc9 : assoc_str2,CREATE TABLE `SMSChannel` (`telNo` varchar(64),`smsProvider` varchar(64),`channelID` int NOT NULL,PRIMARY KEY (`channelID`));CREATE TABLE `ProcessStateMachine` (`stateMachineID` int NOT NULL,PRIMARY KEY (`stateMachineID`));CREATE TABLE `SecEmailChannel` (`secEmailID` int,`channelID` int NOT NULL,KEY `FK_SecEmailChannel_channelID_idx` (`channelID`),PRIMARY KEY (`channelID`));CREATE TABLE `ProcessStateMachineExecution` (`currentStateID` int,`processStateMachineID` int,`processStateMachineExecutionID` int NOT NULL,PRIMARY KEY (`processStateMachineExecutionID`));CREATE TABLE `ProcessStateMachineEvent` (`eventID` int NOT NULL,`stateMachineID` int,KEY `FK_ProcessStateMachineEvent_stateMachineID_idx` (`stateMachineID`),PRIMARY KEY (`eventID`));CREATE TABLE `Channel` (`channelID` int NOT NULL,PRIMARY KEY (`channelID`));CREATE TABLE `ProcessStateMachineTransition` (`toStateID` int,`fromStateID` int,`transitionID` int NOT NULL,PRIMARY KEY (`transitionID`));CREATE TABLE `StateMachineStates` (`stateID` int NOT NULL,`stateMachineID` int NOT NULL,KEY `FK_StateMachineStates_stateID_idx` (`stateID`),KEY `FK_StateMachineStates_stateMachineID_idx` (`stateMachineID`),PRIMARY KEY (`stateID`,`stateMachineID`));CREATE TABLE `PrincipalProxy` (`principalID` int NOT NULL,`channelID` int NOT NULL,KEY `FK_PrincipalProxy_principalID_idx` (`principalID`),KEY `FK_PrincipalProxy_channelID_idx` (`channelID`),PRIMARY KEY (`principalID`,`channelID`));CREATE TABLE `EmailChannel` (`emailID` int,`channelID` int NOT NULL,KEY `FK_EmailChannel_channelID_idx` (`channelID`),PRIMARY KEY (`channelID`));CREATE TABLE `ProcessStateMachineAction` (`actionStateMachineID` int,`actionID` int NOT NULL,PRIMARY KEY (`actionID`));CREATE TABLE `StateMachineTransitions` (`transitionID` int NOT NULL,`stateMachineID` int NOT NULL,KEY `FK_StateMachineTransitions_transitionID_idx` (`transitionID`),KEY `FK_StateMachineTransitions_stateMachineID_idx` (`stateMachineID`),PRIMARY KEY (`transitionID`,`stateMachineID`));CREATE TABLE `ProcessStateMachineState` (`exitActionID` int,`entryActionID` int,`stateID` int NOT NULL,PRIMARY KEY (`stateID`));CREATE TABLE `Principal` (`DType` varchar(64),`principalName` varchar(64),`InstitutionID` int,`period` int,`role` int,`principalID` int NOT NULL,PRIMARY KEY (`principalID`));"
predicted_text_2 = generate_predictions(input_text_2, model, tokenizer)
print("Predicted Output:", predicted_text_2)

Predicted Output: The object-relational strategy represented by this sequence is the Pareto-optimal


In [36]:
# # Test the model with new input data
# import pandas as pd

# # Load CSV file
# df = pd.read_excel("raw_testset_bank.xlsx")  # Update with your file path

# # Assuming 'OM_Regular' is the column name
# texts = df['OM_Regular']

# # Iterate through each text in the column
# for test_text in texts:

#     predicted_text = generate_predictions(test_text, model, tokenizer)
#     print("")
#     print(f"status: {predicted_text}")

In [20]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_curve, roc_auc_score
from sklearn.metrics import precision_recall_curve, classification_report

In [21]:
dc = pd.read_excel('raw_testset_bank.xlsx')

In [22]:
X_test2 = dc['OM_Regular'].values
y_test2 = dc['OM_Prediction'].values

In [23]:
print(X_test2.shape)
print(y_test2.shape)

print("X data type: ", X_test2.dtype)
print("y data type: ", y_test2.dtype)

(32,)
(32,)
X data type:  object
y data type:  int64


In [24]:
print(y_test2)

[1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 1]


In [38]:
dd = pd.read_excel('raw_testset_bank_pred_3.xlsx')

In [39]:
X_test_pred2 = dd['OM_Regular'].values
y_test_pred2 = dd['OM_Prediction'].values

In [40]:
print (y_test_pred2 )

[1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 0 1 1 1 1]


In [41]:
precision = precision_score(y_test2, y_test_pred2)
print("Testing: Precision = %f" % precision)


recall = recall_score(y_test2, y_test_pred2)
print("Testing: Recall = %f" % recall)


f1 = f1_score(y_test2, y_test_pred2)
print("Testing: F1 Score = %f" % f1)

print("\nConfusion Matrix (Test Data):\n", confusion_matrix(y_test2, y_test_pred2))

Testing: Precision = 0.333333
Testing: Recall = 0.900000
Testing: F1 Score = 0.486486

Confusion Matrix (Test Data):
 [[ 4 18]
 [ 1  9]]


In [42]:
print(classification_report(y_test2,y_test_pred2))

              precision    recall  f1-score   support

           0       0.80      0.18      0.30        22
           1       0.33      0.90      0.49        10

    accuracy                           0.41        32
   macro avg       0.57      0.54      0.39        32
weighted avg       0.65      0.41      0.36        32

